In [1]:
# CVE2Patch - CVE Selection
# Cella 1: Importazione delle librerie

import requests
import pandas as pd
import json
import re
import tomllib
import yaml
import hashlib

from pathlib import Path
from datetime import datetime, timezone

print("Librerie importate correttamente.")

Librerie importate correttamente.


In [2]:
# Cella 2: Configurazione del progetto

BASE_DIR = Path.cwd() / "CVE2Patch"

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "output"

# Creazione delle directory
for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Parametri di selezione
START_DATE = "2019-01-01"
END_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

PLATFORM = "linux"
ATTACK_VECTORS = ["NETWORK", "LOCAL"]
EXCLUDE_KERNEL = True

print("Configurazione completata.")
print(f"Periodo di raccolta: {START_DATE} - {END_DATE}")
print(f"Directory del progetto: {BASE_DIR}")

Configurazione completata.
Periodo di raccolta: 2019-01-01 - 2026-11-17
Directory del progetto: d:\leopo\Desktop\Leonardo\Uni\Tesi\Work\CVE\CVE2Patch


In [ ]:
# Cella 3: Acquisizione CVE da NVD

import time
from datetime import timedelta

NVD_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

NVD_DIR = RAW_DIR / "nvd"
NVD_DIR.mkdir(parents=True, exist_ok=True)

start = datetime.fromisoformat(START_DATE)
end = datetime.fromisoformat(END_DATE) + timedelta(days=1)

NVD_FILES = []
total_cves = 0

while start < end:

    stop = min(start + timedelta(days=90), end)
    last = stop - timedelta(milliseconds=1)

    start_index = 0

    while True:

        file_path = NVD_DIR / (
            f"nvd_{start:%Y-%m-%d}_{last:%Y-%m-%d}_{start_index}.json"
        )

        # Se il file esiste, utilizziamo i dati locali
        if file_path.exists():
            data = json.loads(file_path.read_text(encoding="utf-8"))

        else:
            params = {
                "pubStartDate": start.isoformat(timespec="milliseconds"),
                "pubEndDate": last.isoformat(timespec="milliseconds"),
                "resultsPerPage": 2000,
                "startIndex": start_index
            }

            response = requests.get(NVD_URL, params=params, timeout=120)
            response.raise_for_status()

            data = response.json()

            # Salvataggio dei dati originali
            file_path.write_bytes(response.content)

            # Rispetto dei limiti dell'API
            time.sleep(6.5)

        count = len(data["vulnerabilities"])
        total_cves += count
        NVD_FILES.append(file_path)

        start_index += count

        if start_index >= data["totalResults"]:
            break

        if count == 0:
            raise RuntimeError("Paginazione NVD incompleta.")

    print(f"{start:%Y-%m-%d} - {last:%Y-%m-%d} | "
          f"CVE cumulative: {total_cves}")

    start = stop

print(f"\nRaccolta completata: {total_cves} CVE")

2019-01-01 - 2019-03-31 | CVE cumulative: 4156
